<a href="https://colab.research.google.com/github/Kaitokidbua/ASEAN_Transport/blob/main/ASEAN_Part6_EconomicGrowth_Fig21-23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ── Setup ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Dark Theme ────────────────────────────────────────────────────────────────
PAPER_BG   = '#0D1117'
PLOT_BG    = '#161B22'
GRID_C     = '#30363D'
FONT_C     = '#E6EDF3'
MUTED_C    = '#8B949E'
TEMPLATE   = 'plotly_dark'

CITY_COLORS = {
    'Bangkok':      '#FF6B6B',
    'Singapore':    '#74B9FF',
    'Kuala Lumpur': '#4ECDC4',
    'Jakarta':      '#FFB347',
    'Manila':       '#C77DFF',
}

def base_layout(height=460, legend_h=True):
    leg = dict(bgcolor='rgba(22,27,34,0.9)', bordercolor=GRID_C, borderwidth=1, font_size=11)
    if legend_h:
        leg.update(orientation='h', y=1.13, x=1, xanchor='right')
    return dict(
        template=TEMPLATE, paper_bgcolor=PAPER_BG, plot_bgcolor=PLOT_BG,
        font=dict(color=FONT_C, family='Segoe UI, Arial, sans-serif'),
        height=height, legend=leg,
        margin=dict(l=60, r=40, t=75, b=55),
        hoverlabel=dict(bgcolor='#1F2937', bordercolor=GRID_C, font_size=12),
    )

def ax_style():
    return dict(gridcolor=GRID_C, zerolinecolor=GRID_C)

print('✅ Setup complete — Dark theme loaded')

✅ Setup complete — Dark theme loaded


In [10]:
# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('ASEAN_Urban_Growth_Final_with_Mode.csv')
df['Date']  = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# ── Mode label maps ───────────────────────────────────────────────────────────
MODE_BKK = {'BTS':'BTS Skytrain','MRT':'MRT Blue/Purple','SRT':'SRT Red Line',
             'ARL':'Airport Rail Link','YL':'MRT Yellow Line','PK':'MRT Pink Line','RL':'Regional Rail'}
MODE_SGP = {'MRT':'MRT','Public Bus':'Public Bus','LRT':'LRT'}
MODE_KL  = {
    'rail_lrt_kj':'LRT Kelana Jaya','rail_mrt_kajang':'MRT Kajang',
    'rail_lrt_ampang':'LRT Ampang','bus_rkl':'RapidKL Bus',
    'rail_mrt_pjy':'MRT Putrajaya','rail_monorail':'KL Monorail',
    'rail_komuter':'KTM Komuter','rail_komuter_utara':'KTM Utara',
    'rail_ets':'ETS Train','rail_intercity':'Intercity Rail',
    'rail_tebrau':'KTM Tebrau','bus_rkn':'RapidKN Bus','bus_rpn':'RapidPN Bus'
}
MODE_JKT = {'TRANSJAKARTA':'TransJakarta (BRT)','KRL':'KRL Commuter',
             'MRT':'MRT Jakarta','LRT':'LRT Jakarta',
             'BUS SEKOLAH':'School Bus','KCI COMMUTER BANDARA':'Airport Rail','KAPAL':'Ferry'}

print(f'Dataset: {df.shape[0]:,} rows | {df["City"].nunique()} cities')
print('Cities:', df['City'].unique().tolist())

Dataset: 2,027 rows | 5 cities
Cities: ['Bangkok', 'Jakarta', 'Kuala Lumpur', 'Singapore', 'Manila']


In [11]:
# ── Econ prep ─────────────────────────────────────────────────────────────────
yearly = df.groupby(['Year','City'])['Ridership'].sum().reset_index(name='Total_Ridership')
gdp_yr = df.groupby(['Year','City']).agg(
    Total_Ridership=('Ridership','sum'), GDP=('GDP_Billion_USD','mean')
).reset_index()

# ── Chart 6.1: Multi-city Ridership Trend ────────────────────────────────────
c1 = yearly[yearly['Year'].between(2019,2025)]
fig = px.line(c1, x='Year', y='Total_Ridership', color='City',
    markers=True, line_shape='spline',
    title='<b>Fig.21</b>  แนวโน้มผู้โดยสารรายปี ทั้ง 5 เมือง (2019–2025)',
    labels={'Total_Ridership':'ผู้โดยสารรวม','Year':''},
    color_discrete_map=CITY_COLORS)
fig.add_vrect(x0=2019.8, x1=2021.2, fillcolor='#FF4D4D', opacity=0.07,
              annotation_text='⚠ COVID-19', annotation_position='top left',
              annotation_font_color='#FF8080', annotation_font_size=11)
fig.update_layout(**base_layout(480), hovermode='x unified',
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(tickformat='.2s',**ax_style()))
fig.update_traces(line_width=2.5, marker_size=7)
fig.show()



In [12]:
# ── Chart 6.2: YoY All Cities ────────────────────────────────────────────────
yoy_all=[]
for city in yearly['City'].unique():
    sub = yearly[yearly['City']==city].sort_values('Year').copy()
    sub['YoY'] = sub['Total_Ridership'].pct_change()*100
    yoy_all.append(sub)
yoy_df = pd.concat(yoy_all).dropna(subset=['YoY'])
yoy_df = yoy_df[yoy_df['Year'].between(2020,2025)]

fig = px.bar(yoy_df, x='Year', y='YoY', color='City', barmode='group',
    title='<b>Fig.22</b>  อัตราการเปลี่ยนแปลง YoY (%) ทั้ง 5 เมือง',
    labels={'YoY':'YoY Change (%)','Year':''},
    color_discrete_map=CITY_COLORS)
fig.add_hline(y=0, line_dash='dash', line_color=MUTED_C, line_width=1.2)
fig.update_layout(**base_layout(480), hovermode='x unified',
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(ticksuffix='%',**ax_style()))
fig.show()



In [13]:
# ── Chart 6.3: Ridership vs GDP Scatter ──────────────────────────────────────
g3 = gdp_yr[gdp_yr['Year'].between(2021,2024)]
fig = px.scatter(g3, x='GDP', y='Total_Ridership',
    color='City', size='Total_Ridership', size_max=45,
    hover_data=['Year'], text='Year',
    trendline='ols', trendline_scope='trace',
    title='<b>Fig.23</b>  ความสัมพันธ์ระหว่างผู้โดยสารกับ GDP (2021–2024)',
    labels={'GDP':'GDP (Billion USD)','Total_Ridership':'ผู้โดยสารรวม'},
    color_discrete_map=CITY_COLORS)
fig.update_traces(textposition='top center', textfont_size=9,
                  selector=dict(mode='markers+text'))
fig.update_layout(**base_layout(520),
                  xaxis=dict(tickprefix='$',ticksuffix='B',**ax_style()),
                  yaxis=dict(tickformat='.2s',**ax_style()))
fig.show()